# TXT Archive Monitor

Live count of `.txt` files per day in `/Users/msfr/le_monde_archive/txt`.

Run the cell below — it refreshes every 10 seconds until you interrupt the kernel (`■` stop button or **Kernel → Interrupt**).

In [6]:
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cm as cm
from IPython.display import clear_output, display

TXT_ROOT = Path("/Users/msfr/le_monde_archive/txt")


def count_txt_files():
    """Return a DataFrame with one row per YYYY/MM/DD folder and its .txt file count."""
    rows = []
    for day_dir in sorted(TXT_ROOT.glob("*/*/*")):
        parts = day_dir.parts[-3:]  # (YYYY, MM, DD)
        if len(parts) == 3:
            rows.append({
                "date": pd.Timestamp(f"{parts[0]}-{parts[1]}-{parts[2]}"),
                "count": sum(1 for _ in day_dir.glob("*.txt")),
            })
    df = pd.DataFrame(rows, columns=["date", "count"])
    return df.sort_values("date").reset_index(drop=True)


while True:
    df = count_txt_files()
    clear_output(wait=True)

    if df.empty:
        print("No TXT files found yet — waiting for pipeline to produce output…")
    else:
        df["cumsum"] = df["count"].cumsum()
        total = int(df["count"].sum())
        days_done = int((df["count"] > 0).sum())

        # ── viridis color from final cumsum position ───────────────────────
        color = cm.viridis(0.1)

        # ── plot ──────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(16, 8))
        ax.plot(df["date"], df["cumsum"], '.-', linewidth=0.8, color=color)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.xticks(rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Date")
        ax.set_ylabel("Cumulative TXT files")
        ax.set_title(
            f"Cumulative TXT files  —  total: {total:,}  |  days with output: {days_done}"
            f"  |  updated {pd.Timestamp.now():%H:%M:%S}"
        )
        plt.tight_layout()
        plt.show()

        # ── summary table (last 10 days with files) ───────────────────────
        display(
            df[df["count"] > 0]
            .tail(10)[["date", "count", "cumsum"]]
            .assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))
            .reset_index(drop=True)
            .rename(columns={"date": "Date", "count": "TXT files", "cumsum": "Cumulative"})
        )

    time.sleep(10)

KeyboardInterrupt: 